# 3. U-Net segmentation evaluation
Dataset: [LGG MRI Segmentation (Kaggle)](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation) in `data/lgg/kaggle_3m`. Run `python -m scripts.train_unet` first.

In [ ]:
import sys
sys.path.append('..')  # repo root, so models/ and utils/ import
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from utils.preprocessing import load_segmentation_data, split_dataset

In [ ]:
images, masks, patients = load_segmentation_data('../data/lgg/kaggle_3m')
images = images.astype(np.float32) / 255.0
masks = (masks[..., None] > 127).astype(np.float32)
_, _, (x_test, y_test) = split_dataset(images, masks, patients, test_size=0.1, val_size=0.1)
print('test slices:', len(x_test))

In [ ]:
unet = tf.keras.models.load_model('../saved_models/unet_model.keras', compile=False)
pred = (unet.predict(x_test, batch_size=16) > 0.5).astype(np.float32)
t, p = y_test.reshape(len(y_test), -1), pred.reshape(len(pred), -1)
inter = (t * p).sum()
print('global Dice:', 2 * inter / (t.sum() + p.sum()), '| IoU:', inter / (t.sum() + p.sum() - inter))

In [ ]:
idx = np.argsort(-t.sum(1))[:4]
fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for j, i in enumerate(idx):
    for r, m in enumerate([y_test[i], pred[i]]):
        axs[r, j].imshow(x_test[i]); axs[r, j].imshow(np.ma.masked_where(m[..., 0] == 0, m[..., 0]), cmap='autumn', alpha=0.55)
        axs[r, j].set_title(['Ground truth', 'U-Net'][r]); axs[r, j].axis('off')
plt.show()